In [ ]:
from google.colab import drive
drive.mount('/content/drive')

NLP_PATH = '/content/drive/MyDrive/Colab Notebooks/NLP-Costumer-Classifier/'
print("Drive connected!")
print(f"Path: {NLP_PATH}")

In [ ]:
!pip install transformers torch -q
print("Libraries installed!")

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported!")

In [ ]:
with open(NLP_PATH + 'nlp_data.pkl', 'rb') as f:
    data = pickle.load(f)

df = data['df']

print(f"Loaded {len(df):,} complaints")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nCategories:")
print(df['category'].value_counts())

In [ ]:
# Load Sentiment Model
from transformers import pipeline

print("Loading DistilBERT sentiment model...")
print("This downloads once — wait 1-2 minutes...")

sentiment_pipeline = pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english',
    truncation=True,
    max_length=512
)

print(" Sentiment model loaded!")

In [ ]:
# Run Sentiment Analysis
# Sample 1000 for speed
sample = df.sample(min(1000, len(df)), random_state=42).copy()
sample = sample.reset_index(drop=True)

print(f"Analyzing {len(sample):,} complaints...")
print("Wait 5-8 minutes...")

sentiments  = []
scores      = []

for i, text in enumerate(sample['text']):
    try:
        result = sentiment_pipeline(
            str(text)[:512],
            truncation=True
        )[0]
        sentiments.append(result['label'])
        scores.append(round(result['score'], 4))
    except Exception as e:
        sentiments.append('UNKNOWN')
        scores.append(0.0)

    if (i + 1) % 100 == 0:
        print(f"  Processed {i+1}/{len(sample)}...")

sample['sentiment']       = sentiments
sample['sentiment_score'] = scores

print("\n Sentiment analysis complete!")
print(f"\nSentiment distribution:")
print(sample['sentiment'].value_counts())

In [ ]:
# Overall Sentiment Chart
sentiment_counts = sample['sentiment'].value_counts()

colors = {'POSITIVE': '#2ecc71', 'NEGATIVE': '#e74c3c', 'UNKNOWN': '#95a5a6'}
bar_colors = [colors.get(s, '#95a5a6') for s in sentiment_counts.index]

plt.figure(figsize=(8, 5))
sentiment_counts.plot(kind='bar', color=bar_colors, edgecolor='black')
plt.title('Overall Sentiment Distribution', fontweight='bold', fontsize=14)
plt.xlabel('Sentiment')
plt.ylabel('Number of Complaints')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(NLP_PATH + 'sentiment_overall.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Chart saved!")

In [ ]:
# Sentiment by Category Chart
# Filter out UNKNOWN
sample_clean = sample[sample['sentiment'] != 'UNKNOWN'].copy()

sentiment_by_cat = pd.crosstab(
    sample_clean['category'],
    sample_clean['sentiment'],
    normalize='index'
) * 100

print("Sentiment % by Category:")
print(sentiment_by_cat)

sentiment_by_cat.plot(
    kind='bar',
    figsize=(12, 6),
    color=['#e74c3c', '#2ecc71'],
    edgecolor='black'
)
plt.title('Sentiment by Complaint Category', fontweight='bold', fontsize=14)
plt.xlabel('Category')
plt.ylabel('Percentage (%)')
plt.legend(['Negative', 'Positive'], loc='upper right')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(NLP_PATH + 'sentiment_by_category.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Chart saved!")

In [ ]:
# Sentiment Score Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Positive complaints score distribution
pos = sample_clean[sample_clean['sentiment'] == 'POSITIVE']['sentiment_score']
axes[0].hist(pos, bins=20, color='#2ecc71', edgecolor='black')
axes[0].set_title('Positive Sentiment Score Distribution', fontweight='bold')
axes[0].set_xlabel('Confidence Score')
axes[0].set_ylabel('Count')

# Negative complaints score distribution
neg = sample_clean[sample_clean['sentiment'] == 'NEGATIVE']['sentiment_score']
axes[1].hist(neg, bins=20, color='#e74c3c', edgecolor='black')
axes[1].set_title('Negative Sentiment Score Distribution', fontweight='bold')
axes[1].set_xlabel('Confidence Score')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(NLP_PATH + 'sentiment_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved!")

In [ ]:
# Top Negative Keywords Per Category
from collections import Counter
import re

print("Top negative words by category:\n")

for cat in sample_clean['category'].unique():
    neg_texts = sample_clean[
        (sample_clean['category'] == cat) &
        (sample_clean['sentiment'] == 'NEGATIVE')
    ]['clean_text'].tolist()

    if not neg_texts:
        continue

    all_words = ' '.join(neg_texts).split()
    top_words = Counter(all_words).most_common(8)

    print(f"  {cat}:")
    print(f"  {[w[0] for w in top_words]}")
    print()

In [ ]:
# Summary Stats
print("="*50)
print("SENTIMENT ANALYSIS SUMMARY")
print("="*50)

total     = len(sample_clean)
positive  = len(sample_clean[sample_clean['sentiment'] == 'POSITIVE'])
negative  = len(sample_clean[sample_clean['sentiment'] == 'NEGATIVE'])

print(f"Total analyzed:  {total:,}")
print(f"Positive:        {positive:,} ({positive/total*100:.1f}%)")
print(f"Negative:        {negative:,} ({negative/total*100:.1f}%)")

print(f"\nBy Category:")
for cat in sample_clean['category'].unique():
    cat_df   = sample_clean[sample_clean['category'] == cat]
    neg_pct  = len(cat_df[cat_df['sentiment'] == 'NEGATIVE']) / len(cat_df) * 100
    print(f"  {cat:<25} {neg_pct:.1f}% negative")

print("="*50)

In [ ]:
# Save sentiment results
sample.to_csv(NLP_PATH + 'sentiment_results.csv', index=False)

# Save updated df with sentiment back to pkl
with open(NLP_PATH + 'nlp_data.pkl', 'rb') as f:
    data = pickle.load(f)

data['sentiment_sample'] = sample

with open(NLP_PATH + 'nlp_data.pkl', 'wb') as f:
    pickle.dump(data, f)

print("="*50)
print("✅ NOTEBOOK 4 COMPLETE!")
print("="*50)
print("\nFiles saved to Drive:")
print("  ✅ sentiment_results.csv")
print("  ✅ sentiment_overall.png")
print("  ✅ sentiment_by_category.png")
print("  ✅ sentiment_score_distribution.png")
print("  ✅ nlp_data.pkl (updated)")
print("\nNow open Notebook 5 — Streamlit Dashboard")